# Day 12: LangChain (Part 2)

## Goal

Connect LangChain to a vector database and build a basic retrieval flow.

By the end of this notebook you will have:

- a small document set
- embeddings stored in Chroma
- a retriever that finds relevant context
- a basic RAG pipeline that answers questions from retrieved documents

This notebook extends Day 11 by adding retrieval.

## Step 0: Sync the Project Once

The Day 12 packages are already included in `pyproject.toml`, so learners only need:

```bash
uv sync
```

Set the values you want in `.env`.

Example:

```text
OPENAI_API_KEY=your_openai_key
OPENAI_MODEL=your_openai_model
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
OLLAMA_MODEL=llama3.2
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_EMBEDDING_MODEL=nomic-embed-text
```

If you plan to use Ollama, make sure the server is running locally and the chat and embedding models are already pulled.

For course maintainers, the Day 12 integration package was added with:

```bash
uv add langchain-chroma
```

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

print("Imports loaded successfully.")


In [ ]:
load_dotenv(override=True)

config_summary = {
    "OPENAI_API_KEY_present": bool(os.getenv("OPENAI_API_KEY")),
    "OPENAI_MODEL": os.getenv("OPENAI_MODEL"),
    "OPENAI_EMBEDDING_MODEL": os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    "OLLAMA_MODEL": os.getenv("OLLAMA_MODEL"),
    "OLLAMA_BASE_URL": os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
    "OLLAMA_EMBEDDING_MODEL": os.getenv("OLLAMA_EMBEDDING_MODEL"),
}

config_summary

## Step 1: Create a Small Knowledge Base

We will store a few short documents about invoice processing so the retrieval behavior is easy to inspect.

In [ ]:
documents = [
    Document(
        page_content="Invoices usually contain vendor name, invoice date, invoice number, due date, line items, taxes, and total amount.",
        metadata={"source": "invoice_fields"},
    ),
    Document(
        page_content="Before payment, the finance team validates invoice totals, confirms the vendor, and checks approval status.",
        metadata={"source": "finance_checks"},
    ),
    Document(
        page_content="Travel reimbursement claims should include receipts, travel dates, and manager approval before submission.",
        metadata={"source": "travel_policy"},
    ),
    Document(
        page_content="Late payment risk can be reduced by tracking due dates and matching invoices with approved purchase records.",
        metadata={"source": "payment_risk"},
    ),
]

documents

## Step 2: Create Helpers for the Chat Model and Embeddings

This notebook supports either:

- OpenAI for chat + embeddings
- Ollama for chat + embeddings

In [ ]:
def get_chat_model(provider: str):
    if provider == "openai":
        api_key = os.getenv("OPENAI_API_KEY")
        model = os.getenv("OPENAI_MODEL")
        if not api_key:
            raise ValueError("OPENAI_API_KEY is not set in .env")
        if not model:
            raise ValueError("OPENAI_MODEL is not set in .env")
        return ChatOpenAI(model=model, api_key=api_key)

    if provider == "ollama":
        model = os.getenv("OLLAMA_MODEL")
        if not model:
            raise ValueError("OLLAMA_MODEL is not set in .env")
        return ChatOllama(
            model=model,
            base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
        )

    raise ValueError("provider must be 'openai' or 'ollama'")


def get_embedding_model(provider: str):
    if provider == "openai":
        api_key = os.getenv("OPENAI_API_KEY")
        model = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
        if not api_key:
            raise ValueError("OPENAI_API_KEY is not set in .env")
        return OpenAIEmbeddings(model=model, api_key=api_key)

    if provider == "ollama":
        model = os.getenv("OLLAMA_EMBEDDING_MODEL")
        if not model:
            raise ValueError("OLLAMA_EMBEDDING_MODEL is not set in .env")
        return OllamaEmbeddings(
            model=model,
            base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
        )

    raise ValueError("provider must be 'openai' or 'ollama'")


print("Provider helpers ready.")

## Step 3: Build the Chroma Vector Store

This turns the document set into a searchable vector database.

In [ ]:
provider = "openai"  # Change to "ollama" if you want to use Ollama.

embedding_model = get_embedding_model(provider)
db_path = Path("day12/chroma_store")
db_path.mkdir(parents=True, exist_ok=True)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=str(db_path),
    collection_name="day12_training_docs",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Vector store and retriever ready.")

## Step 4: Test Retrieval Without an LLM

This is useful because it lets you inspect whether retrieval is working before you involve generation.

In [ ]:
question = "What fields are normally present in an invoice, and what does finance verify before payment?"
question

In [ ]:
retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"Document {i}")
    print("Source:", doc.metadata["source"])
    print(doc.page_content)
    print("-" * 80)

## Step 5: Build a Basic RAG Helper

The pattern is:
- retrieve relevant documents
- format them as context
- send the question + context to the model


In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chat_model = get_chat_model(provider)

rag_prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant. Answer the user's question using only the context below.
If the answer is not in the context, say you do not know.

Context:
{context}

Question:
{question}
    """.strip()
)


def answer_question(question: str) -> str:
    docs = retriever.invoke(question)
    context = format_docs(docs)
    messages = rag_prompt.format_messages(context=context, question=question)
    response = chat_model.invoke(messages)
    return response.content


print("RAG helper ready.")


In [ ]:
print(answer_question(question))


## Step 6: Try Another Question

Change the question below and run the next cell.

Examples:

- `How can late payment risk be reduced?`
- `What does a travel reimbursement claim require?`
- `What should finance validate before paying a vendor?`

In [ ]:
print(answer_question("How can late payment risk be reduced?"))


## Day 12 Recap

You now have a basic RAG pipeline with LangChain:

- documents were embedded and stored in Chroma
- a retriever found relevant context
- a prompt combined the retrieved context with the user question
- the LLM generated an answer from that context

This is the core retrieval flow before moving to larger document ingestion.